# Delta Lake MERGE Implementation using Azure Databricks

## Objective

This notebook demonstrates how to implement Delta Lake MERGE operations using Delta Lake.

### Workflow

- Load customer datasets
- Explore the data
- Create a Delta table
- Perform SCD Type 1 MERGE
- Perform SCD Type 2 MERGE
- Validate the results

In [0]:
# Load the uploaded tables

master_df = spark.table("default.customer_master_cleaned")
increment_df = spark.table("default.customer_incremental_cleaned")

display(master_df)

display(increment_df)

+-------------+
|      catalog|
+-------------+
|my_databricks|
|      samples|
|       system|
+-------------+

+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
+------------------+



In [0]:
spark.sql("SHOW TABLES IN my_databricks.default").show(truncate=False)

+--------+----------------------------+-----------+
|database|tableName                   |isTemporary|
+--------+----------------------------+-----------+
|default |customer_incremental_cleaned|false      |
|default |customer_master_cleaned     |false      |
+--------+----------------------------+-----------+



In [0]:
# Load the uploaded tables

master_df = spark.table("my_databricks.default.customer_master_cleaned")
increment_df = spark.table("my_databricks.default.customer_incremental_cleaned")

display(master_df)
display(increment_df)

customer_id,customer_name,city,age,product,quantity,price,total_amount
1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0
1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0
1003,Sneha,Nashik,45,Mouse,5,10317.0,51585.0
1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0
1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0
1006,Neha,Unknown,51,Laptop,4,42076.0,168304.0
1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290.0
1008,Amit,Pune,30,Headphones,2,3767.0,7534.0
1009,Nikhil,Delhi,36,Keyboard,4,17709.0,70836.0
1010,Aadhya,Indore,57,Tablet,2,24552.0,49104.0


customer_id,customer_name,city,age,product,quantity,price,total_amount
1003,Aditya,Pune,31,Laptop,2,60000,120000
1012,Sneha,Mumbai,29,Phone,1,30000,30000
1025,Rahul,Delhi,40,Tablet,3,25000,75000
1040,Meera,Nagpur,36,Keyboard,1,4000,4000
1055,Aarav,Hyderabad,27,Mouse,4,1500,6000
1101,Yash,Surat,25,Laptop,2,45000,90000
1102,Tanvi,Lucknow,24,Phone,1,22000,22000
1103,Om,Pune,28,Tablet,2,18000,36000
1104,Siya,Mumbai,30,Laptop,3,50000,150000
1105,Ved,Goa,26,Mouse,1,2000,2000


In [0]:
print("Master Records:", master_df.count())
print("Increment Records:", increment_df.count())

Master Records: 100
Increment Records: 10


# Step 1: Create Delta Table

The cleaned master dataset is stored in Delta format. Delta Lake provides ACID transactions, schema enforcement, and supports MERGE operations.

In [0]:
# Save the master dataset as a Delta table

master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_delta")

In [0]:
delta_df = spark.table("customer_delta")

display(delta_df)

customer_id,customer_name,city,age,product,quantity,price,total_amount
1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0
1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0
1003,Sneha,Nashik,45,Mouse,5,10317.0,51585.0
1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0
1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0
1006,Neha,Unknown,51,Laptop,4,42076.0,168304.0
1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290.0
1008,Amit,Pune,30,Headphones,2,3767.0,7534.0
1009,Nikhil,Delhi,36,Keyboard,4,17709.0,70836.0
1010,Aadhya,Indore,57,Tablet,2,24552.0,49104.0


In [0]:
print("Total Records:", delta_df.count())

Total Records: 100


# Step 2: Perform SCD Type 1 using Delta MERGE

SCD Type 1 updates existing records and inserts new records without maintaining history.

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "customer_delta")

(
    delta_table.alias("target")
    .merge(
        increment_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_name": "source.customer_name",
            "city": "source.city",
            "age": "source.age",
            "product": "source.product",
            "quantity": "source.quantity",
            "price": "source.price",
            "total_amount": "source.total_amount"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "city": "source.city",
            "age": "source.age",
            "product": "source.product",
            "quantity": "source.quantity",
            "price": "source.price",
            "total_amount": "source.total_amount"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.table("customer_delta"))

customer_id,customer_name,city,age,product,quantity,price,total_amount
1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0
1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0
1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0
1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0
1006,Neha,Unknown,51,Laptop,4,42076.0,168304.0
1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290.0
1008,Amit,Pune,30,Headphones,2,3767.0,7534.0
1009,Nikhil,Delhi,36,Keyboard,4,17709.0,70836.0
1010,Aadhya,Indore,57,Tablet,2,24552.0,49104.0
1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039.0


In [0]:
print("Records after MERGE:", spark.table("customer_delta").count())

Records after MERGE: 105


# Step 3: Implement SCD Type 2

SCD Type 2 preserves historical data by creating a new version of a record whenever changes occur.

Additional columns used:
- is_current
- start_date
- end_date

In [0]:
from pyspark.sql.functions import lit, current_date

# Create SCD2 table from current Delta table
scd2_df = spark.table("customer_delta") \
    .withColumn("is_current", lit(True)) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date"))

scd2_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_delta_scd2")

display(spark.table("customer_delta_scd2"))

customer_id,customer_name,city,age,product,quantity,price,total_amount,is_current,start_date,end_date
1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0,true,2026-07-31,null
1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0,true,2026-07-31,null
1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0,true,2026-07-31,null
1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0,true,2026-07-31,null
1006,Neha,Unknown,51,Laptop,4,42076.0,168304.0,true,2026-07-31,null
1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290.0,true,2026-07-31,null
1008,Amit,Pune,30,Headphones,2,3767.0,7534.0,true,2026-07-31,null
1009,Nikhil,Delhi,36,Keyboard,4,17709.0,70836.0,true,2026-07-31,null
1010,Aadhya,Indore,57,Tablet,2,24552.0,49104.0,true,2026-07-31,null
1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039.0,true,2026-07-31,null


In [0]:
spark.table("customer_delta_scd2").printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)



# Step 4: Validation

Validate the Delta tables after implementing SCD Type 1 and SCD Type 2.

In [0]:
print("Customer Delta Records:", spark.table("customer_delta").count())
print("Customer Delta SCD2 Records:", spark.table("customer_delta_scd2").count())

Customer Delta Records: 105
Customer Delta SCD2 Records: 105


In [0]:
%sql
SELECT *
FROM customer_delta
LIMIT 10;

customer_id,customer_name,city,age,product,quantity,price,total_amount
1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0
1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0
1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0
1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0
1006,Neha,Unknown,51,Laptop,4,42076.0,168304.0
1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290.0
1008,Amit,Pune,30,Headphones,2,3767.0,7534.0
1009,Nikhil,Delhi,36,Keyboard,4,17709.0,70836.0
1010,Aadhya,Indore,57,Tablet,2,24552.0,49104.0
1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039.0


In [0]:
%sql
SELECT *
FROM customer_delta_scd2
WHERE is_current = true;

customer_id,customer_name,city,age,product,quantity,price,total_amount,is_current,start_date,end_date
1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0,true,2026-07-31,null
1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0,true,2026-07-31,null
1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0,true,2026-07-31,null
1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0,true,2026-07-31,null
1006,Neha,Unknown,51,Laptop,4,42076.0,168304.0,true,2026-07-31,null
1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290.0,true,2026-07-31,null
1008,Amit,Pune,30,Headphones,2,3767.0,7534.0,true,2026-07-31,null
1009,Nikhil,Delhi,36,Keyboard,4,17709.0,70836.0,true,2026-07-31,null
1010,Aadhya,Indore,57,Tablet,2,24552.0,49104.0,true,2026-07-31,null
1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039.0,true,2026-07-31,null


In [0]:
display(
    spark.table("customer_delta_scd2")
         .filter("is_current = true")
)

customer_id,customer_name,city,age,product,quantity,price,total_amount,is_current,start_date,end_date
1001,Krishna,Chennai,38,Phone,5,49549.0,247745.0,true,2026-07-31,null
1002,Kunal,Chennai,21,Keyboard,4,15805.0,63220.0,true,2026-07-31,null
1004,Aadhya,Pune,51,Laptop,4,11761.0,47044.0,true,2026-07-31,null
1005,Ishaan,Chennai,25,Headphones,3,917.0,2751.0,true,2026-07-31,null
1006,Neha,Unknown,51,Laptop,4,42076.0,168304.0,true,2026-07-31,null
1007,Krishna,Nashik,23,Keyboard,5,20858.0,104290.0,true,2026-07-31,null
1008,Amit,Pune,30,Headphones,2,3767.0,7534.0,true,2026-07-31,null
1009,Nikhil,Delhi,36,Keyboard,4,17709.0,70836.0,true,2026-07-31,null
1010,Aadhya,Indore,57,Tablet,2,24552.0,49104.0,true,2026-07-31,null
1011,Aadhya,Hyderabad,43,Headphones,3,39013.0,117039.0,true,2026-07-31,null


# Conclusion

This assignment demonstrated the implementation of Delta Lake using Azure Databricks.

### Completed Tasks

- Created customer datasets using Python
- Cleaned data using Pandas
- Uploaded datasets to Azure Databricks
- Created Delta tables
- Implemented SCD Type 1 using MERGE
- Created an SCD Type 2 table with historical tracking columns
- Validated the final results

The notebook demonstrates the use of Delta Lake for efficient data management and Slowly Changing Dimension handling.